# News dataset preprocessing

Cleans a scraped-news dataset with columns `source, title, url, date, author, content` using pandas, and keeps
the original next to the cleaned version. Run the cells in order.

| # | Function | Purpose |
|---|---|---|
| 1 | `load()` | read the xlsx/csv as text, normalize headers |
| 2 | `clean_text()` | shared base cleaner (HTML, unicode, quotes, whitespace) |
| 3 | `clean_source()` | one canonical name per outlet |
| 4 | `clean_title()` | headline cleanup |
| 5 | `clean_author()` | byline cleanup |
| 6 | `clean_url()` | canonical URLs |
| 7 | `clean_content()` | strip boilerplate, fix glued sentences |
| 8 | `parse_dates()` | mixed formats/timezones to UTC |
| 9 | `preprocess()` | before/after table and drop flags |
| 10 | `report()` + saving | summary and the two output files |

## Setup

In [13]:
import html
import re
import unicodedata
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

import pandas as pd

INPUT_FILE = "news_articles3.csv"          # .csv or .xlsx
COMPARISON_FILE = "news_before_after.csv"  # all rows: original + cleaned columns
CLEAN_FILE = "news_clean.csv"              # kept rows only, ready for the DB

## 1 · `load()` : read the raw file

**What it does:** reads your `.xlsx` or `.csv`, normalizes the column names, checks that the six required
columns exist, and returns a DataFrame where *every value is kept as text*.

**Why `dtype=str, keep_default_na=False`:** it stops pandas from "helping". Without it, an author literally
called `NA` becomes a missing value, and dates, ids or leading zeros can be silently converted. The raw data
is kept exactly as scraped, so the *original* columns in the before/after file are trustworthy.

In [14]:
COLUMNS = ["source", "title", "url", "date", "author", "content"]


def load(path):
    p = Path(path)
    if p.suffix.lower() == ".csv":
        df = pd.read_csv(p, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    else:
        df = pd.read_excel(p, dtype=str, keep_default_na=False)
    df.columns = [str(c).strip().lower() for c in df.columns]
    missing = [c for c in COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in file: {missing}. Found: {list(df.columns)}")
    return df[COLUMNS].reset_index(drop=True)

## 2 · `clean_text()` : the shared base cleaner

**What it does:** the generic cleanup that every text column goes through first.

1. `html.unescape` : `&amp;` → `&`, `&#39;` → `'`
2. strip HTML tags : `<p>`, `<b>` …
3. `unicodedata.normalize("NFKC")` : non-breaking spaces → normal spaces, ligatures `ﬁ` → `fi`, full-width letters → normal, `…` → `...`
4. remove zero-width / soft-hyphen characters
5. curly quotes → straight quotes
6. collapse any run of whitespace (spaces, tabs, newlines) into one space and trim
7. empty / whitespace-only / `None` / `NaN` → `None`

`clean_source`, `clean_title`, `clean_author`, `clean_url` and `clean_content` all start by calling this function.

In [15]:
TAG_RE = re.compile(r"<[^>]+>")
WS_RE = re.compile(r"\s+")
ZERO_WIDTH = dict.fromkeys(map(ord, "\u200b\u200c\u200d\u2060\ufeff\u00ad"), None)
QUOTES = {
    ord("\u2018"): "'", ord("\u2019"): "'",
    ord("\u201c"): '"', ord("\u201d"): '"',
}


def clean_text(value):
    """Generic text cleanup: HTML, unicode, quotes, whitespace. Returns None if empty."""
    if value is None or pd.isna(value):
        return None
    s = html.unescape(str(value))
    s = TAG_RE.sub(" ", s)
    s = unicodedata.normalize("NFKC", s)      # also turns non-breaking spaces into spaces
    s = s.translate(ZERO_WIDTH).translate(QUOTES)
    s = WS_RE.sub(" ", s).strip()
    return s or None

## 3 · `clean_source()` : one canonical name per outlet

**What it does:** runs `clean_text`, then looks the lower-cased name up in `SOURCE_MAP`. Known variants become one
canonical spelling; unknown sources are kept as they are (just cleaned).

**Why:** `RTE`, `rté news` and `RTE News` would otherwise be three different values when you `GROUP BY source`.
Add your own outlets to `SOURCE_MAP`.

In [16]:
# Edit freely: lower-case variant -> canonical name
SOURCE_MAP = {
    "rte": "RTE",
    "rté": "RTE",
    "rte news": "RTE",
    "the verge": "The Verge",
    "verge": "The Verge",
    "evening standard": "Evening Standard",
    "the guardian": "The Guardian",
    "guardian": "The Guardian",
}


def clean_source(value):
    s = clean_text(value)
    return SOURCE_MAP.get(s.casefold(), s) if s else None

## 4 · `clean_title()` : headline cleanup

**What it does:** applies `clean_text` to the headline (HTML entities, tags, odd unicode, quotes, spacing).
It is its own function so you can later add title-specific rules (for example stripping a trailing ` | RTE`)
without touching the other columns.

In [17]:
def clean_title(value):
    return clean_text(value)

## 5 · `clean_author()` : byline cleanup

**What it does:**

1. `clean_text`
2. removes a leading `By`, `Written by` or `Words by`
3. trims stray commas, semicolons and dashes
4. placeholders (`unknown`, `n/a`, `staff`, `none`, `-` …) → `None`
5. ALL CAPS names → Title Case

In [18]:
AUTHOR_PLACEHOLDERS = {"", "n/a", "na", "none", "null", "unknown", "staff", "-"}


def clean_author(value):
    s = clean_text(value)
    if not s:
        return None
    s = re.sub(r"^(written by|words by|by)\s+", "", s, flags=re.I)
    s = s.strip(" ,;-")
    if s.casefold() in AUTHOR_PLACEHOLDERS:
        return None
    if s.isupper():
        s = s.title()
    return s or None

## 6 · `clean_url()` : canonical URLs

**What it does:**

1. `clean_text`, then remove any spaces inside the URL
2. keep only `http` / `https` URLs with a host; anything else → `None`
3. lower-case the scheme and host (the path keeps its case, since paths are case-sensitive)
4. drop tracking query parameters (`utm_*`, `fbclid`, `gclid`, `mc_*`, `cmp*`) but keep real ones like `?id=42`
5. drop the `#fragment` and the trailing `/`

**Why:** the same article often appears under several URLs. Canonical URLs make duplicate detection (and a `UNIQUE (url)` constraint) reliable.

In [19]:
TRACKING_PREFIXES = ("utm_", "fbclid", "gclid", "mc_", "cmp")


def clean_url(value):
    s = clean_text(value)
    if not s:
        return None
    s = s.replace(" ", "")
    parts = urlsplit(s)
    if parts.scheme.lower() not in ("http", "https") or not parts.netloc:
        return None
    query = [
        (k, v)
        for k, v in parse_qsl(parts.query, keep_blank_values=True)
        if not k.lower().startswith(TRACKING_PREFIXES)
    ]
    path = parts.path.rstrip("/") or "/"
    return urlunsplit(
        (parts.scheme.lower(), parts.netloc.lower(), path, urlencode(query), "")
    )
# HTTPS://WWW.RTE.IE/news/a/?utm_source=x&id=42#comments
# → https://www.rte.ie/news/a?id=42

## 7 · `clean_content()` : article body cleanup

**What it does:**

1. `clean_text` (HTML, unicode, quotes, whitespace)
2. removes scraped boilerplate with the regexes in `BOILERPLATE_RE`
   (the *Exco Player cookie notice* and The Verge's *"may earn a commission"* line)
3. re-inserts the missing space where sentences were glued together: `week.The man` → `week. The man`
4. collapses whitespace again and returns `None` if nothing is left

To handle another recurring junk phrase, add one more `re.compile(...)` to `BOILERPLATE_RE`.

In [20]:
# Boilerplate that gets scraped along with the article text
BOILERPLATE_RE = [
    re.compile(
        r"This content is provided by [^.]*?\.\s*Please click ['\"]?Allow and Continue['\"]? "
        r"below to load the content\.?",
        re.I,
    ),
    re.compile(
        r"If you buy something from a link,? .{0,40}? may earn a commission\."
        r"(?: See our ethics statement\.?)?",
        re.I,
    ),
]

# "...last week.The man..." -> "...last week. The man..."
SENTENCE_GLUE_RE = re.compile(r"(?<=[a-z0-9])([.!?])(?=[A-Z][a-z])")


def clean_content(value):
    s = clean_text(value)
    if not s:
        return None
    for pattern in BOILERPLATE_RE:
        s = pattern.sub(" ", s)
    s = SENTENCE_GLUE_RE.sub(r"\1 ", s)
    s = WS_RE.sub(" ", s).strip()
    return s or None

## 8 · `parse_dates()` : mixed formats and timezones → one UTC column

**What it does:** `pd.to_datetime(..., errors="coerce", utc=True, format="mixed")`

- `utc=True` : converts every timezone to UTC, so a mix of `+00:00` and `+05:00` no longer raises
  *"Mixed timezones detected"*
- `format="mixed"` : parses each value on its own. Without it pandas guesses one format from the first
  value and turns every value that does not match into `NaT`
- `errors="coerce"` : garbage such as `not a date` becomes `NaT` (→ `NULL` in the DB) instead of crashing

Requires pandas 2.0 or newer.

In [21]:
def parse_dates(series):
    """Parse mixed formats / timezones and normalise everything to UTC."""
    return pd.to_datetime(series, errors="coerce", utc=True, format="mixed")

## 9 · `preprocess()` : build the before/after table and flag bad rows

**What it does:**

1. adds `row_id` (the row number as it appears in Excel)
2. for each column keeps `<col>_original` (untouched) next to `<col>_clean`, using the cleaners from sections 2 to 8
3. adds `content_chars_before` / `content_chars_after`
4. flags rows to drop, **first matching rule wins**:

| order | `drop_reason` | rule |
|---|---|---|
| 1 | `missing_or_invalid_url` | cleaned URL is empty |
| 2 | `missing_content` | cleaned content is empty |
| 3 | `content_too_short` | fewer than `MIN_CONTENT_CHARS` (100) characters |
| 4 | `duplicate_url` | same cleaned URL as an earlier kept row |
| 5 | `duplicate_content` | same cleaned content as an earlier kept row |

Rows are **flagged, not deleted**, so the comparison file keeps every original row.
Duplicates are only searched among rows that are still "alive", so a row that was dropped for another reason can never make a good row look like a duplicate.

In [22]:
MIN_CONTENT_CHARS = 100                   # shorter articles are dropped


def preprocess(df):
    cleaners = {
        "source": clean_source,
        "title": clean_title,
        "url": clean_url,
        "author": clean_author,
        "content": clean_content,
    }

    out = pd.DataFrame({"row_id": range(2, len(df) + 2)})   # row number as seen in Excel
    for col in COLUMNS:
        out[f"{col}_original"] = df[col]                    # untouched copy
        if col == "date":
            out["date_clean"] = parse_dates(df["date"])
        else:
            out[f"{col}_clean"] = df[col].map(cleaners[col])

    out["content_chars_before"] = out["content_original"].str.len()
    out["content_chars_after"] = out["content_clean"].str.len()

    # ---- decide which rows to drop (rows are flagged, never deleted from the comparison file)
    reason = pd.Series("", index=out.index)

    def mark(mask, label):
        reason.loc[mask & reason.eq("")] = label

    def mark_duplicates(col, label):
        alive = reason.eq("")
        dup = out.loc[alive, col].duplicated(keep="first")
        reason.loc[dup[dup].index] = label

    mark(out["url_clean"].isna(), "missing_or_invalid_url")
    mark(out["content_clean"].isna(), "missing_content")
    mark(out["content_clean"].str.len().lt(MIN_CONTENT_CHARS), "content_too_short")
    mark_duplicates("url_clean", "duplicate_url")
    mark_duplicates("content_clean", "duplicate_content")

    out["status"] = reason.map(lambda r: "kept" if r == "" else "dropped")
    out["drop_reason"] = reason
    return out

## 10 · `report()` and saving the two output files

**`report(out)`** prints a summary: rows read / kept / dropped, a count per drop reason, and how many cells
each column's cleaning actually changed. It is the quickest way to see whether the cleaning is doing something sensible on your full 75k-row file.

**Saving** writes two files (UTF-8 with BOM, so Excel shows accents correctly):

| file | contents |
|---|---|
| `news_before_after.csv` | **all** rows, `<col>_original` + `<col>_clean`, status, drop_reason |
| `news_clean.csv` | **kept** rows only, with the original column names, ready for PostgreSQL |

CSV is used instead of `.xlsx` because an Excel cell holds at most 32,767 characters and long articles can exceed that.

In [23]:
def report(out):
    print(f"Rows read           : {len(out):,}")
    print(f"Rows kept           : {(out['status'] == 'kept').sum():,}")
    print(f"Rows dropped        : {(out['status'] == 'dropped').sum():,}")
    print(out.loc[out["drop_reason"] != "", "drop_reason"].value_counts().to_string())
    print("\nCells changed by cleaning:")
    for col in ["source", "title", "url", "author", "content"]:
        before = out[f"{col}_original"].fillna("").astype(str)
        after = out[f"{col}_clean"].fillna("").astype(str)
        print(f"  {col:<8}: {(before != after).sum():,}")
    print(f"  date     : {out['date_clean'].isna().sum():,} unparseable (stored as empty)")

## Run

In [24]:
df = load(INPUT_FILE)
out = preprocess(df)
report(out)

# all rows, original + cleaned columns side by side
out.to_csv(COMPARISON_FILE, index=False, encoding="utf-8-sig")

# kept rows only, original column names, ready for PostgreSQL
clean = out.loc[out["status"] == "kept", [f"{c}_clean" for c in COLUMNS]].copy()
clean.columns = COLUMNS
clean.to_csv(CLEAN_FILE, index=False, encoding="utf-8-sig")

Rows read           : 78,720
Rows kept           : 78,121
Rows dropped        : 599
drop_reason
duplicate_content    599

Cells changed by cleaning:
  source  : 0
  title   : 13,865
  url     : 0
  author  : 7,198
  content : 78,689
  date     : 7,425 unparseable (stored as empty)
